In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("OSUHackathon2025_TeamGameData (1).csv")

BAD_MISSING = ['#DIV/0!', 'NA', 'na', '', ' ']
df = df.replace(BAD_MISSING, np.nan)

keep_cols = [
    "Season", "GameId", "Date",
    "Team", "OppTeam",
    "Score", "OppScore",
    # raw stats for rolling features
    "OffensiveEPAPerPlay_All",
    "OffensiveSuccessRate_All",
    "OffensiveBoomRate_All",
    "OffensiveBustRate_All",
    "Sum_PasserPoints",
    "Sum_RusherPoints",
    "Sum_ReceiverPoints",
    "Sum_PassBlockPoints",
    "Sum_RunBlockPoints",
    "Sum_PassCovPoints",
    "Sum_PassRushPoints",
    "Sum_RunDefPoints"
]

df = df[keep_cols].copy()

num_cols = [
    "Season", "GameId", "Score", "OppScore",
    "OffensiveEPAPerPlay_All", "OffensiveSuccessRate_All",
    "OffensiveBoomRate_All", "OffensiveBustRate_All", "Sum_PasserPoints",
    "Sum_RusherPoints", "Sum_ReceiverPoints", "Sum_PassBlockPoints",
    "Sum_RunBlockPoints", "Sum_PassCovPoints", "Sum_PassRushPoints",
    "Sum_RunDefPoints"
]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["Season", "GameId", "Team", "OppTeam", "Score", "OppScore", "Date"])
df["ScoreDiff"] = df["Score"] - df["OppScore"]
df["Outcome"] = (df["ScoreDiff"] > 0).astype(int)

df["Date_dt"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date_dt"])

print("Shape after Step 1:", df.shape)
print("Outcome distribution:\n", df["Outcome"].value_counts())
print("Date range:", df["Date_dt"].min(), "->", df["Date_dt"].max())


Shape after Step 1: (10791, 22)
Outcome distribution:
 Outcome
0    5396
1    5395
Name: count, dtype: int64
Date range: 2026-03-31 00:00:00 -> 2026-03-31 00:00:00


In [ ]:
df = df.sort_values(["Season", "Team", "Date_dt"]).reset_index(drop=True)
rolling_cols = [
    "OffensiveEPAPerPlay_All",
    "OffensiveSuccessRate_All",
    "OffensiveBoomRate_All",
    "OffensiveBustRate_All",
    "Sum_PasserPoints",
    "Sum_RusherPoints",
    "Sum_ReceiverPoints",
    "Sum_PassBlockPoints",
    "Sum_RunBlockPoints",
    "Sum_PassCovPoints",
    "Sum_PassRushPoints",
    "Sum_RunDefPoints"
]

for col in rolling_cols:
    df[f"{col}_last5"] = (
        df.groupby(["Season", "Team"])[col]
          .transform(lambda x: x.shift(1).rolling(1).mean())
    )

print("Rolling features created.")

Rolling features created.


In [ ]:
opp_df = df[[
    "Season", "Team", "Date_dt",
] + [f"{c}_last5" for c in rolling_cols]].copy()


opp_df = opp_df.rename(columns={
    "Team": "OppTeam",
    **{f"{c}_last5": f"{c}_opp_last5" for c in rolling_cols}
})

df = df.merge(
    opp_df,
    on=["Season", "OppTeam", "Date_dt"],
    how="left"
)

df["Diff_last5_EPA"] = df["OffensiveEPAPerPlay_All_last5"] - df["OffensiveEPAPerPlay_All_opp_last5"]
df["Diff_last5_SR"]  = df["OffensiveSuccessRate_All_last5"] - df["OffensiveSuccessRate_All_opp_last5"]
df["Diff_last5_BoR"] = df["OffensiveBoomRate_All_last5"] - df["OffensiveBoomRate_All_opp_last5"]
df["Diff_last5_BuR"] = df["OffensiveBustRate_All_last5"] - df["OffensiveBustRate_All_opp_last5"]
df["Diff_last5_PP"] = df["Sum_PasserPoints_last5"] - df["Sum_PasserPoints_opp_last5"]
df["Diff_last5_RuP"] = df["Sum_RusherPoints_last5"] - df["Sum_RusherPoints_opp_last5"]
df["Diff_last5_ReP"] = df["Sum_ReceiverPoints_last5"] - df["Sum_ReceiverPoints_opp_last5"]
df["Diff_last5_PBP"] = df["Sum_PassBlockPoints_last5"] - df["Sum_PassBlockPoints_opp_last5"]
df["Diff_last5_RBP"] = df["Sum_RunBlockPoints_last5"] - df["Sum_RunBlockPoints_opp_last5"]
df["Diff_last5_PCP"] = df["Sum_PassCovPoints_last5"] - df["Sum_PassCovPoints_opp_last5"]
df["Diff_last5_PRP"] = df["Sum_PassRushPoints_last5"] - df["Sum_PassRushPoints_opp_last5"]
df["Diff_last5_RDP"] = df["Sum_RunDefPoints_last5"] - df["Sum_RunDefPoints_opp_last5"]

feature_cols = [
    "Diff_last5_EPA",
    "Diff_last5_SR",
    "Diff_last5_BoR",
    "Diff_last5_BuR",
    "Diff_last5_PP",
    "Diff_last5_RuP",
    "Diff_last5_ReP",
    "Diff_last5_PBP",
    "Diff_last5_RBP",
    "Diff_last5_PCP",
    "Diff_last5_PRP",
    "Diff_last5_RDP"
]

# feature_cols = [
#    "OffensiveEPAPerPlay_All_last5",
#    "OffensiveSuccessRate_All_last5",
#    "OffensiveBoomRate_All_last5",
#    "OffensiveBustRate_All_last5",
#    "Sum_PasserPoints_last5",
#    "Sum_RusherPoints_last5",
#    "Sum_ReceiverPoints_last5",
#    "Sum_PassBlockPoints_last5",
#    "Sum_RunBlockPoints_last5",
#    "Sum_PassCovPoints_last5",
#    "Sum_PassRushPoints_last5",
#    "Sum_RunDefPoints_last5"
# ]

print("Diff features created.")
df_model = df.dropna(subset=feature_cols).copy()

print("Rows before:", len(df))
print("Rows after :", len(df_model))


Diff features created.
Rows before: 10791
Rows after : 9092


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# feature_cols = [
#    "OffensiveEPAPerPlay_All_last5",
#    "OffensiveSuccessRate_All_last5",
#    "OffensiveBoomRate_All_last5",
#    "OffensiveBustRate_All_last5",
#    "Sum_PasserPoints_last5",
#    "Sum_RusherPoints_last5",
#    "Sum_ReceiverPoints_last5",
#    "Sum_PassBlockPoints_last5",
#    "Sum_RunBlockPoints_last5",
#    "Sum_PassCovPoints_last5",
#    "Sum_PassRushPoints_last5",
#    "Sum_RunDefPoints_last5"
# ]

X = df_model[feature_cols].apply(pd.to_numeric, errors="coerce")
y = df_model["Outcome"].astype(int)

# Safety: if any NaN still exists in X (should be rare), drop those rows
mask_ok = ~X.isna().any(axis=1)
X = X.loc[mask_ok].copy()
y = y.loc[mask_ok].copy()

print("Final modeling shape:", X.shape, "Outcome balance:", y.value_counts().to_dict())

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# MLP
# model = Pipeline([
#    ("imputer", SimpleImputer(strategy="median")),   # safe even if no NaN
#    ("scaler", StandardScaler()),
#    ("mlp", MLPClassifier(
#        hidden_layer_sizes=(64, 32),   # small model to start; you can tune later
#        activation="relu",
#        solver="adam",
#        alpha=1e-4,
#        learning_rate_init=1e-3,
#        max_iter=300,
#        random_state=42,
#        early_stopping=True,
#        n_iter_no_change=10,
#        verbose=False
#    ))
# ])

# SVM
from sklearn.svm import SVC

model = Pipeline([
     ('imputer', SimpleImputer(strategy='median')),
     ('scaler', StandardScaler()),
     ('svm', SVC(
         kernel='rbf',
         C=1.0,
         gamma='scale',
         probability=True,
         random_state=42
     ))
])


model.fit(X_train, y_train)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"[Classification] Accuracy={acc:.4f}  ROC-AUC={auc:.4f}\n")
print(classification_report(y_test, y_pred, digits=3))

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}, Features used: {X_train.shape[1]}")


Final modeling shape: (9092, 12) Outcome balance: {0: 4546, 1: 4546}
[Classification] Accuracy=0.6267  ROC-AUC=0.6599

              precision    recall  f1-score   support

           0      0.628     0.624     0.626       910
           1      0.626     0.629     0.628       909

    accuracy                          0.627      1819
   macro avg      0.627     0.627     0.627      1819
weighted avg      0.627     0.627     0.627      1819

Training samples: 7273, Test samples: 1819, Features used: 12
